# MA2006B: Curvas Elípticas sobre ℝ, 𝔽ₚ y ECDH

**Integrantes del equipo:**
- Gerardo Javier Lopez Garcia - A01660262
- Chiara Bombardieri Balanza - A01659462
- Emilio Guillen - 

---

## Parte 1 — Curvas Elípticas sobre los Reales

### ¿Qué es una curva elíptica?

Una **curva elíptica** es el conjunto de puntos $(x, y)$ que satisfacen la ecuación de Weierstrass corta:

$$E : y^2 = x^3 + ax + b$$

junto con un punto especial llamado **punto al infinito**, denotado $\mathcal{O}$.

Para que la curva tenga una estructura algebraica bien definida (sin cúspides ni auto-intersecciones), se requiere que sea **no singular**. Esto se verifica con el **discriminante**:

$$\Delta = 4a^3 + 27b^2$$

Si $\Delta = 0$, la curva es singular y no podemos usarla.

---

### El punto al infinito

El **punto al infinito** $\mathcal{O} = (\text{None}, \text{None})$ es el **elemento neutro** del grupo de puntos. Es el análogo al cero en la suma ordinaria:

$$P + \mathcal{O} = \mathcal{O} + P = P \quad \text{para todo punto } P$$

Geométricamente, se puede imaginar como el punto donde todas las rectas verticales se "encuentran" en el infinito.

---

### ¿Qué significa sumar dos puntos?

La **ley de grupo** en una curva elíptica define cómo sumar dos puntos $P$ y $Q$. Geométricamente:

1. Se traza una recta que pasa por $P$ y $Q$.
2. Esa recta intersecta la curva en un tercer punto.
3. Se refleja ese tercer punto sobre el eje $x$: el resultado es $P + Q$.

Algebraicamente, se calcula una pendiente $m$ y luego:

$$x_3 = m^2 - x_1 - x_2 \qquad y_3 = m(x_1 - x_3) - y_1$$

Hay varios casos importantes:
- **Identidad**: si uno de los puntos es $\mathcal{O}$, el resultado es el otro punto.
- **Inversos**: si $P = (x, y)$ y $Q = (x, -y)$, la recta es vertical y no corta la curva en un tercer punto finito; el resultado es $\mathcal{O}$.
- **Duplicación** ($P = Q$): se usa la recta tangente en $P$, con pendiente $m = \frac{3x_1^2 + a}{2y_1}$.
- **Puntos distintos** ($P \neq Q$): pendiente ordinaria $m = \frac{y_2 - y_1}{x_2 - x_1}$.

---

### ¿Por qué se usa tolerancia con floats?

Los números de punto flotante (`float` en Python) son representaciones **aproximadas** de los reales. Las operaciones aritméticas acumulan pequeños errores de redondeo. Por ejemplo:

```python
>>> 0.1 + 0.2 == 0.3
False
>>> 0.1 + 0.2
0.30000000000000004
```

Por eso, en lugar de comparar con `==`, usamos `math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-9)`, que devuelve `True` si la diferencia entre `a` y `b` es suficientemente pequeña. Esto hace que todas las validaciones y comparaciones de coordenadas sean **robustas** frente a errores de redondeo.

In [1]:
import math


class EllipticCurveReal:
    """
    Curva elíptica sobre los números reales (usando float).

    La curva sigue la forma de Weierstrass corta:
        E : y² = x³ + ax + b

    El punto al infinito se representa como (None, None).
    """

    def __init__(self, a: float, b: float) -> None:
        """
        Inicializa la curva con coeficientes a y b.

        Parámetros
        ----------
        a : float
            Coeficiente lineal de la curva.
        b : float
            Término independiente de la curva.

        Lanza
        -----
        ValueError
            Si el discriminante Δ = 4a³ + 27b² es prácticamente cero,
            lo que indica que la curva es singular.
        """
        self.a = float(a)
        self.b = float(b)

        # Discriminante: la curva es válida sólo si Δ ≠ 0
        delta = 4 * self.a**3 + 27 * self.b**2
        if abs(delta) < 1e-9:
            raise ValueError("La curva es singular")

    def is_on_curve(self, P: tuple) -> bool:
        """
        Comprueba si el punto P pertenece a la curva.

        Parámetros
        ----------
        P : tuple
            Par (x, y) de floats, o (None, None) para el punto al infinito.

        Retorna
        -------
        bool
            True si P está sobre la curva, False en caso contrario.
        """
        # El punto al infinito siempre pertenece a la curva
        if P == (None, None):
            return True

        x, y = P
        left_side = y**2
        right_side = x**3 + self.a * x + self.b

        # Usamos math.isclose para comparación robusta con floats
        return math.isclose(left_side, right_side, rel_tol=1e-9, abs_tol=1e-9)

    def add_points(self, P: tuple, Q: tuple) -> tuple:
        """
        Suma dos puntos de la curva usando la ley de grupo.

        Parámetros
        ----------
        P : tuple
            Primer punto (x1, y1) o (None, None).
        Q : tuple
            Segundo punto (x2, y2) o (None, None).

        Retorna
        -------
        tuple
            El punto resultante de P + Q.

        Lanza
        -----
        ValueError
            Si alguno de los puntos no está sobre la curva.
        """
        # Validar que ambos puntos están sobre la curva
        if not self.is_on_curve(P):
            raise ValueError("El punto no está sobre la curva")
        if not self.is_on_curve(Q):
            raise ValueError("El punto no está sobre la curva")

        # Caso 1 — Identidad: cualquier punto + O = ese punto
        if P == (None, None):
            return Q
        if Q == (None, None):
            return P

        x1, y1 = P
        x2, y2 = Q

        # Caso 2 — Inversos: P y Q tienen la misma x pero y opuestas → O
        # Se usa tolerancia para comparaciones de float
        if math.isclose(x1, x2, rel_tol=1e-9, abs_tol=1e-9) and \
           math.isclose(y1, -y2, rel_tol=1e-9, abs_tol=1e-9):
            return (None, None)

        # Caso 3 — Duplicación: P == Q, se usa la tangente
        if math.isclose(x1, x2, rel_tol=1e-9, abs_tol=1e-9) and \
           math.isclose(y1, y2, rel_tol=1e-9, abs_tol=1e-9):
            m = (3 * x1**2 + self.a) / (2 * y1)
        else:
            # Caso 4 — Puntos distintos: pendiente ordinaria
            m = (y2 - y1) / (x2 - x1)

        # Coordenadas del punto resultante
        x3 = m**2 - x1 - x2
        y3 = m * (x1 - x3) - y1

        return (x3, y3)

### Prueba rápida (Test Case 1 del enunciado)

Curva $E : y^2 = x^3 - x + 1$ con `a = -1.0`, `b = 1.0`.

In [2]:
curve_real = EllipticCurveReal(a=-1.0, b=1.0)

assert curve_real.is_on_curve((1.0, 1.0)) == True
assert curve_real.add_points((1.0, 1.0), (1.0, 1.0)) == (-1.0, 1.0)

print("Parte 1 OK:", curve_real.add_points((1.0, 1.0), (1.0, 1.0)))

Parte 1 OK: (-1.0, 1.0)


## Parte 2: Curvas elípticas sobre campos finitos

En esta sección implementamos la aritmética de curvas elípticas sobre un campo finito \(\mathbb{F}_p\), donde \(p\) es un número primo. La curva tiene la misma forma de Weierstrass corta:

\[
E: y^2 = x^3 + ax + b \pmod p
\]

A diferencia del caso real, todas las operaciones se realizan módulo \(p\). Esto significa que cada coordenada y cada coeficiente se reduce con `% p`.

El punto al infinito se representa como:

```python
(None, None)
```

In [12]:
class EllipticCurveFp:
    """ 
    Curva elíptica sobre el campo primo 𝔽_p.

    La curva sigue la forma de Weierstrass corta:
        E : y^2 = x^3 + a x + b (mod p)

    El punto al infinito se representa como (None, None).
    """ 

    def __init__(self, a: int, b: int, p: int) -> None:
        if p <= 2:
            raise ValueError("p debe ser mayor que 2")

        self.p = p
        self.a = a % p
        self.b = b % p

        delta = (4 * self.a**3 + 27 * self.b**2) % p

        if delta == 0:
            raise ValueError("La curva es singular módulo p")

    def mod_inverse(self, k: int, p: int) -> int:
        k = k % p

        if k == 0:
            raise ZeroDivisionError("No existe inverso")

        a, b = k, p
        x0, x1 = 1, 0

        while b != 0:
            q = a // b

            a, b = b, a - q * b
            x0, x1 = x1, x0 - q * x1

        if a != 1:
            raise ZeroDivisionError("No existe inverso")

        return x0 % p

    def is_on_curve(self, P: tuple[int, int] | tuple[None, None]) -> bool:
        if P == (None, None):
            return True

        x, y = P

        left = (y * y) % self.p
        right = (x * x * x + self.a * x + self.b) % self.p

        return left == right

    def add_points(self, P: tuple, Q: tuple) -> tuple:
        if not self.is_on_curve(P):
            raise ValueError("El punto no está sobre la curva")

        if not self.is_on_curve(Q):
            raise ValueError("El punto no está sobre la curva")

        if P == (None, None):
            return Q

        if Q == (None, None):
            return P

        x1, y1 = P
        x2, y2 = Q
        p = self.p

        if x1 == x2 and (y1 + y2) % p == 0:
            return (None, None)

        if P == Q:
            m = ((3 * x1**2 + self.a) * self.mod_inverse(2 * y1, p)) % p
        else:
            m = ((y2 - y1) * self.mod_inverse(x2 - x1, p)) % p

        x3 = (m**2 - x1 - x2) % p
        y3 = (m * (x1 - x3) - y1) % p

        return (x3, y3)

    def scalar_multiply(self, k: int, P: tuple) -> tuple:
        if not self.is_on_curve(P):
            raise ValueError("El punto no está sobre la curva")

        if k == 0 or P == (None, None):
            return (None, None)

        if k < 0:
            x, y = P
            P = (x, (-y) % self.p)
            k = -k

        result = (None, None)
        addend = P

        while k > 0:
            if k & 1:
                result = self.add_points(result, addend)

            addend = self.add_points(addend, addend)
            k >>= 1

        return result

In [11]:
def simulate_ecdh(curve: EllipticCurveFp, G: tuple, d_A: int, d_B: int) -> tuple:
    Q_A = curve.scalar_multiply(d_A, G)
    Q_B = curve.scalar_multiply(d_B, G)

    S_A = curve.scalar_multiply(d_A, Q_B)
    S_B = curve.scalar_multiply(d_B, Q_A)

    assert S_A == S_B
    return S_A

In [10]:
curve_fp = EllipticCurveFp(a=2, b=2, p=17)

assert curve_fp.is_on_curve((5, 1)) == True
assert curve_fp.add_points((5, 1), (5, 1)) == (6, 3)
assert curve_fp.add_points((5, 1), (10, 6)) == (3, 1)
assert curve_fp.scalar_multiply(10, (5, 1)) == (7, 11)
assert curve_fp.scalar_multiply(19, (5, 1)) == (None, None)

Q_A = curve_fp.scalar_multiply(3, (5, 1))
Q_B = curve_fp.scalar_multiply(7, (5, 1))

assert Q_A == (10, 6)
assert Q_B == (0, 6)

assert simulate_ecdh(curve_fp, (5, 1), 3, 7) == (6, 3)

print("Parte 2 OK: EllipticCurveFp y ECDH funcionan correctamente")

Parte 2 OK: EllipticCurveFp y ECDH funcionan correctamente
